In [ ]:
#Import Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Libraries loaded successfully!')

## IPUMS - Employed_Only

In [ ]:
# Load IPUMS dataset
ipums_path = os.path.expanduser('~/repoad688-employability-sum26-groupB/data/processed/employed_only.csv')
ipums = pd.read_csv(ipums_path)
print(f'Shape: {ipums.shape}')
print(f'\nColumns: {ipums.columns.tolist()}')
ipums.head(10)

In [ ]:
# Summary Statustics
print('Summary Statistics:')
ipums.describe(include='all')

In [ ]:
# Handle missing values
missing = ipums.isnull().sum()
missing_pct = (missing / len(ipums) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)
print(f'Columns with missing values: {len(missing_df)} of {ipums.shape[1]}')
missing_df

### Wage Gap by Gender

In [ ]:
# Gender Distribution
print('Gender Distribution:')
print(ipums['SEX_LABEL'].value_counts())
print()
print('Gender % Distribution:')
print((ipums['SEX_LABEL'].value_counts(normalize=True) * 100).round(2))

In [ ]:
# Breakdown of $0 wage
zero_wage = ipums[ipums['INCWAGE'] == 0]
print(f'Records with $0 wage: {len(zero_wage):,}')
print(f'Percentage: {(len(zero_wage) / len(ipums) * 100):.2f}%')
print()
print('Gender breakdown of $0 wage:')
print(zero_wage['SEX_LABEL'].value_counts())

In [ ]:
print('Average wage INCLUDING $0:')
print(ipums.groupby('SEX_LABEL')['INCWAGE'].mean().round(2))

print('\nAverage wage EXCLUDING $0:')
ipums_nonzero = ipums[ipums['INCWAGE'] > 0]
print(ipums_nonzero.groupby('SEX_LABEL')['INCWAGE'].mean().round(2))
print(f'\nRows remaining after removing $0: {len(ipums_nonzero):,}')

In [ ]:
# Remove $0 wage (outliers)
ipums_clean = ipums[ipums['INCWAGE'] > 0].copy()
print(f'Clean dataset shape: {ipums_clean.shape}')
print(f'Removed {len(ipums) - len(ipums_clean):,} rows with $0 wage')

In [ ]:
# Visulaize wage gap by gender
avg_wage = ipums_clean.groupby('SEX_LABEL')['INCWAGE'].mean().reset_index()

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(avg_wage['SEX_LABEL'], avg_wage['INCWAGE'],
              color=['tomato', 'steelblue'], edgecolor='black', width=0.5)

ax.set_title('Average Wage by Gender (Employed, 2024)', fontsize=14)
ax.set_xlabel('Gender')
ax.set_ylabel('Average Annual Wage ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

for bar, val in zip(bars, avg_wage['INCWAGE']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'${val:,.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
#
clean_path = os.path.expanduser('~/repoad688-employability-sum26-groupB/data/clean/')
os.makedirs(clean_path, exist_ok=True)

output_path = f'{clean_path}employed_only_clean.csv'
ipums_clean.to_csv(output_path, index=False)
print(f'Saved to: {output_path}')
print(f'Shape: {ipums_clean.shape}')

### Wage Gap by State

In [ ]:
# Wage gap by state
state_wage = ipums_clean.groupby(['STATE_NAME', 'SEX_LABEL'])['INCWAGE'].mean().unstack()
state_wage['GAP'] = state_wage['Male'] - state_wage['Female']
state_wage['WOMEN_PCT'] = (state_wage['Female'] / state_wage['Male'] * 100).round(1)
state_wage = state_wage.sort_values('GAP', ascending=False)

print('Top 10 states with largest wage gap:')
state_wage.head(10)

In [ ]:
# Top 15 states with largest wage gap
top15 = state_wage.head(15)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top15.index, top15['GAP'], color='steelblue', edgecolor='black')

ax.set_title('Top 15 States with Largest Gender Wage Gap (2024)', fontsize=14)
ax.set_xlabel('Wage Gap (Male - Female) $')
ax.set_ylabel('State')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

for bar, val in zip(bars, top15['GAP']):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

# Lightcast - Occupation

In [ ]:
lc_occ = pd.read_csv('/home/ubuntu/repoad688-employability-sum26-groupB/data/processed/lightcast_occupation.csv')
print(f'Shape: {lc_occ.shape}')
print(f'Columns: {lc_occ.columns.tolist()}')
lc_occ.head(10)

In [ ]:
# Check unique values in each column
print(f'Shape: {lc_occ.shape}')
print(f'\nUnique values per column:')
for col in lc_occ.columns:
    print(f'  {col}: {lc_occ[col].nunique()} unique values')

In [ ]:
print('LOT_V6_CAREER_AREA_NAME values:')
print(lc_occ['LOT_V6_CAREER_AREA_NAME'].value_counts())

print('\nLOT_V6_OCCUPATION_GROUP_NAME values:')
print(lc_occ['LOT_V6_OCCUPATION_GROUP_NAME'].value_counts())

In [ ]:
# Missing values check
missing = lc_occ.isnull().sum()
missing_pct = (missing / len(lc_occ) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)
print(f'Columns with missing values: {len(missing_df)}')
missing_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Career Area Distribution
lc_occ['LOT_V6_CAREER_AREA_NAME'].value_counts().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='black'
)
axes[0].set_title('Job Postings by Career Area', fontsize=13)
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of Job Postings')
axes[0].tick_params(axis='x', rotation=30)

# Occupation Group Distribution
lc_occ['LOT_V6_OCCUPATION_GROUP_NAME'].value_counts().plot(
    kind='barh', ax=axes[1], color='tomato', edgecolor='black'
)
axes[1].set_title('Job Postings by Occupation Group', fontsize=13)
axes[1].set_xlabel('Number of Job Postings')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Check IPUMS occupation codes
print('IPUMS OCC unique values (sample):')
print(ipums_clean['OCC'].value_counts().head(10))

print('\nIPUMS IND unique values (sample):')
print(ipums_clean['IND'].value_counts().head(10))

In [ ]:
# Check Lightcast SOC codes
print('Lightcast SOC_5 unique values:')
print(lc_occ['SOC_5'].value_counts())

print('\nLightcast LOT_V6_OCCUPATION_GROUP_NAME:')
print(lc_occ['LOT_V6_OCCUPATION_GROUP_NAME'].value_counts())

In [ ]:
# IT-related OCC codes in IPUMS are typically 1000-1999
ipums_it = ipums_clean[(ipums_clean['OCC'] >= 1000) & (ipums_clean['OCC'] <= 1999)]
print(f'IT workers in IPUMS: {len(ipums_it):,}')
print(f'\nGender distribution in IT:')
print(ipums_it['SEX_LABEL'].value_counts())
print(f'\nGender % in IT:')
print((ipums_it['SEX_LABEL'].value_counts(normalize=True) * 100).round(2))
print(f'\nAverage wage by gender in IT:')
print(ipums_it.groupby('SEX_LABEL')['INCWAGE'].mean().round(2))

In [ ]:
# Compare IT vs Overall workforce gender gap
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Chart 1: Gender distribution - Overall vs IT
categories = ['Overall Workforce', 'IT Workforce']
male_pct = [51.77, 72.42]
female_pct = [48.23, 27.58]

x = range(len(categories))
axes[0].bar(x, male_pct, width=0.4, label='Male', color='steelblue', edgecolor='black')
axes[0].bar([i + 0.4 for i in x], female_pct, width=0.4, label='Female', color='tomato', edgecolor='black')
axes[0].set_title('Gender Distribution:\nOverall vs IT Workforce', fontsize=12)
axes[0].set_ylabel('Percentage (%)')
axes[0].set_xticks([i + 0.2 for i in x])
axes[0].set_xticklabels(categories)
axes[0].legend()

# Chart 2: Average wage by gender - Overall
avg_overall = ipums_clean.groupby('SEX_LABEL')['INCWAGE'].mean()
axes[1].bar(avg_overall.index, avg_overall.values, color=['tomato', 'steelblue'], edgecolor='black')
axes[1].set_title('Average Wage by Gender:\nOverall Workforce', fontsize=12)
axes[1].set_ylabel('Average Annual Wage ($)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, v in enumerate(avg_overall.values):
    axes[1].text(i, v + 500, f'${v:,.0f}', ha='center', fontweight='bold', fontsize=9)

# Chart 3: Average wage by gender - IT
avg_it = ipums_it.groupby('SEX_LABEL')['INCWAGE'].mean()
axes[2].bar(avg_it.index, avg_it.values, color=['tomato', 'steelblue'], edgecolor='black')
axes[2].set_title('Average Wage by Gender:\nIT Workforce', fontsize=12)
axes[2].set_ylabel('Average Annual Wage ($)')
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, v in enumerate(avg_it.values):
    axes[2].text(i, v + 500, f'${v:,.0f}', ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Box plot 1: Overall workforce wage distribution by gender
ipums_clean.boxplot(column='INCWAGE', by='SEX_LABEL', ax=axes[0],
                    patch_artist=True,
                    boxprops=dict(facecolor='steelblue', color='black'),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='o', markersize=2, alpha=0.3))

axes[0].set_title('Wage Distribution by Gender\nOverall Workforce', fontsize=12)
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Annual Wage ($)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].set_ylim(0, 300000)

# Box plot 2: IT workforce wage distribution by gender
ipums_it.boxplot(column='INCWAGE', by='SEX_LABEL', ax=axes[1],
                 patch_artist=True,
                 boxprops=dict(facecolor='tomato', color='black'),
                 medianprops=dict(color='blue', linewidth=2),
                 flierprops=dict(marker='o', markersize=2, alpha=0.3))

axes[1].set_title('Wage Distribution by Gender\nIT Workforce', fontsize=12)
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Annual Wage ($)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].set_ylim(0, 300000)

plt.suptitle('')
plt.tight_layout()
plt.show()

# Visuals using Plotly

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import os

FIGURES_DIR = '/home/ubuntu/repoad688-employability-sum26-groupB/assets/figures/'

def save_fig(fig, name):
    fig.write_image(f"{FIGURES_DIR}{name}.png")
    print(f"Saved {name}.png")

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import os

FIGURES_DIR = '/home/ubuntu/repoad688-employability-sum26-groupB/assets/figures/'

def save_fig(fig, name):
    fig.write_image(f"{FIGURES_DIR}{name}.png")
    print(f"Saved {name}.png")

print('Plotly loaded successfully!')

## Gender Distribution: Overall vs IT

In [ ]:
# Data
categories = ['Overall Workforce', 'IT Workforce']
male_pct = [51.77, 72.42]
female_pct = [48.23, 27.58]

fig1 = go.Figure()
fig1.add_trace(go.Bar(name='Male', x=categories, y=male_pct, marker_color='steelblue'))
fig1.add_trace(go.Bar(name='Female', x=categories, y=female_pct, marker_color='tomato'))

fig1.update_layout(
    barmode='group',
    title='Gender Distribution: Overall vs IT Workforce (2024)',
    xaxis_title='Workforce Type',
    yaxis_title='Percentage (%)',
    legend_title='Gender',
    template='plotly_white',
    height=500
)
save_fig(fig1, 'antara_gender_distribution')
fig1.show()

## Average Wage by Gender: Overall vs IT

In [ ]:
fig2 = go.Figure()

# Overall
fig2.add_trace(go.Bar(
    name='Overall - Female', x=['Overall'], y=[59562],
    marker_color='tomato', offsetgroup=0
))
fig2.add_trace(go.Bar(
    name='Overall - Male', x=['Overall'], y=[84342],
    marker_color='steelblue', offsetgroup=1
))

# IT
fig2.add_trace(go.Bar(
    name='IT - Female', x=['IT'], y=[96398],
    marker_color='tomato', offsetgroup=0,
    showlegend=False
))
fig2.add_trace(go.Bar(
    name='IT - Male', x=['IT'], y=[119699],
    marker_color='steelblue', offsetgroup=1,
    showlegend=False
))

fig2.update_layout(
    barmode='group',
    title='Average Wage by Gender: Overall vs IT Workforce (2024)',
    xaxis_title='Workforce Type',
    yaxis_title='Average Annual Wage ($)',
    template='plotly_white',
    height=500
)
save_fig(fig2, 'antara_avg_wage_comparison')
fig2.show()

## Box Plot Wage Distribution

In [ ]:
# Combine overall and IT data
ipums_clean['Workforce'] = 'Overall'
ipums_it_plot = ipums_it.copy()
ipums_it_plot['Workforce'] = 'IT'
combined = pd.concat([ipums_clean, ipums_it_plot])

fig3 = px.box(
    combined[combined['INCWAGE'] <= 300000],
    x='Workforce',
    y='INCWAGE',
    color='SEX_LABEL',
    title='Wage Distribution by Gender: Overall vs IT Workforce (2024)',
    labels={'INCWAGE': 'Annual Wage ($)', 'SEX_LABEL': 'Gender', 'Workforce': 'Workforce Type'},
    color_discrete_map={'Male': 'steelblue', 'Female': 'tomato'},
    template='plotly_white',
    height=550
)
save_fig(fig3, 'antara_wage_boxplot')
fig3.show()

## Lightcast Career Area

In [ ]:
career_counts = lc_occ['LOT_V6_CAREER_AREA_NAME'].value_counts().reset_index()
career_counts.columns = ['Career Area', 'Count']

fig4 = px.bar(
    career_counts,
    x='Count',
    y='Career Area',
    orientation='h',
    title='Job Postings by Career Area (Lightcast 2024)',
    labels={'Count': 'Number of Job Postings', 'Career Area': ''},
    color='Count',
    color_continuous_scale='Blues',
    template='plotly_white',
    height=400
)
save_fig(fig4, 'antara_career_area')
fig4.show()

## Lightcast Occupation Group

In [ ]:
occ_counts = lc_occ['LOT_V6_OCCUPATION_GROUP_NAME'].value_counts().reset_index()
occ_counts.columns = ['Occupation Group', 'Count']

fig5 = px.bar(
    occ_counts,
    x='Count',
    y='Occupation Group',
    orientation='h',
    title='Job Postings by Occupation Group (Lightcast 2024)',
    labels={'Count': 'Number of Job Postings', 'Occupation Group': ''},
    color='Count',
    color_continuous_scale='Reds',
    template='plotly_white',
    height=450
)
save_fig(fig5, 'antara_occupation_group')
fig5.show()